# Logistic regression tutorial

We train **binary** `mlpackage.supervised_learning.LogisticRegression` (batch gradient descent on cross-entropy) on sklearn's **Wisconsin Breast Cancer** data: **569** samples, **30** numeric features, labels **malignant** vs **benign** (encoded 0/1). Feature **standardization** (fit on train only) helps stable gradient steps.

## Prerequisites and goals

**You will**
- Load a two-class problem with continuous features.
- Split train/test with **stratification** so both classes appear in each fold.
- Standardize using **training** mean/variance, then fit logistic regression.
- Evaluate **accuracy**, inspect **predicted probabilities**, and compare **iteration counts** on the same split.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `mlpackage`.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from mlpackage.supervised_learning import LogisticRegression

## Step 1: Load Breast Cancer data

Targets: **0 = malignant**, **1 = benign** (sklearn ordering). The feature matrix is already numeric; we will scale it before optimization.

In [ ]:
cancer = load_breast_cancer()
X = np.asarray(cancer.data, dtype=float)
y = np.asarray(cancer.target, dtype=int)

feature_names = list(cancer.feature_names)
target_names = list(cancer.target_names)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class names:", {i: target_names[i] for i in range(2)})
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))
print("\nFirst feature row (raw):", np.round(X[0], 3))

## Step 2: Stratified train/test split

With two classes, stratification keeps minority/majority rates similar in train and test (here the classes are only mildly imbalanced, but the habit is good practice).

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape[0], "  Test:", X_test.shape[0])
print("Train counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test counts :", dict(zip(*np.unique(y_test, return_counts=True))))

## Step 3: Standardize features (train-only fit)

Gradient descent can converge faster when inputs share comparable scales. **`StandardScaler`** learns mean/variance from **`X_train`** only.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Standardized train shape:", X_train_s.shape)

## Step 4: Fit `LogisticRegression`

Defaults (`learning_rate=0.01`, `n_iterations=1000`) often work on scaled tabular data; adjust if loss plateaus slowly.

In [ ]:
LEARNING_RATE = 0.1
N_ITER = 2000

clf = LogisticRegression(learning_rate=LEARNING_RATE, n_iterations=N_ITER)
clf.fit(X_train_s, y_train.astype(float))

print("Bias b:", round(clf.bias, 4))
w = pd.Series(clf.weights, index=feature_names, name="weights").sort_values(key=abs, ascending=False)
display(w.head(10).round(4))

## Step 5: Predict classes and probabilities on the test set

- **`predict_probability`**: \(\hat{p}_i = P(y_i=1\mid \mathbf{x}_i)\)
- **`predict`**: threshold at **0.5** (as implemented in the package)

In [ ]:
p_test = clf.predict_probability(X_test_s)
y_pred = clf.predict(X_test_s)

n_show = 12
preview = pd.DataFrame(
    {
        "y_true": [target_names[t] for t in y_test[:n_show]],
        "P(benign)": np.round(p_test[:n_show], 4),
        "y_pred": [target_names[t] for t in y_pred[:n_show]],
    }
)
display(preview)

## Step 6: Accuracy and confusion-style counts

`score` is **accuracy**; we also show per-class **true positive** style tallies for the 2×2 structure.

In [ ]:
print(f"Train accuracy: {clf.score(X_train_s, y_train):.4f}")
print(f"Test accuracy : {clf.score(X_test_s, y_test):.4f}")

for name, y_t, y_p in [
    ("Test", y_test, y_pred),
]:
    for true_label in (0, 1):
        mask = y_t == true_label
        correct = int(np.sum((y_p == y_t) & mask))
        total = int(np.sum(mask))
        print(
            f"{name} true {target_names[true_label]}: {correct}/{total} correct"
        )

## Step 7: Visualize two standardized features

We plot **mean radius** vs **mean texture** (first two features) for all samples after applying the **same** scaler. This is a 2D slice of a 30D problem; the learned hyperplane lives in the full space.

In [ ]:
X_all_s = scaler.transform(X)

i, j = 0, 1
colors = ["tab:blue", "tab:orange"]

plt.figure(figsize=(7, 5))
for klass in (0, 1):
    m = y == klass
    plt.scatter(
        X_all_s[m, i],
        X_all_s[m, j],
        c=colors[klass],
        label=target_names[klass],
        alpha=0.8,
        edgecolors="black",
        linewidths=0.3,
        s=40,
    )
plt.xlabel(f"{feature_names[i]} (standardized)")
plt.ylabel(f"{feature_names[j]} (standardized)")
plt.title("Breast cancer: two features (StandardScaler)")
plt.legend()
plt.tight_layout()

from pathlib import Path

_nb_here = Path("logistic_regression_tutorial.ipynb")
if _nb_here.is_file():
    _plot_path = _nb_here.with_name("breast_cancer_two_features_scatter.png")
else:
    _plot_path = Path(
        "examples/supervised_learning/logistic_regression/breast_cancer_two_features_scatter.png"
    )
    _plot_path.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_plot_path.resolve()}")
plt.show()

## Step 8: Compare `n_iterations` (same split, same `learning_rate`)

More iterations can improve fit up to a point; if learning rate is too large, training can become unstable. Here we only vary **`n_iterations`**.

In [ ]:
iters = [200, 500, 1000, 2000, 5000]
rows = []

for n in iters:
    m = LogisticRegression(learning_rate=LEARNING_RATE, n_iterations=n)
    m.fit(X_train_s, y_train.astype(float))
    rows.append(
        {
            "n_iterations": n,
            "train_acc": m.score(X_train_s, y_train),
            "test_acc": m.score(X_test_s, y_test),
        }
    )

display(pd.DataFrame(rows).round(4))